# 🎬 Universal Video LoRA Trainer (Wan 2.1 & Wan 2.2)
Bộ công cụ huấn luyện LoRA Video Đa Năng (Text-to-Video & Image-to-Video) cho:
- **Wan 2.1**: Wan2.1-T2V-14B, Wan2.1-I2V-14B-720P, Wan2.1-I2V-14B-480P, Wan2.1-T2V-1.3B
- **Wan 2.2**: Wan2.2-T2V-14B, Wan2.2-I2V-14B

### ☕ Bước 1: Khởi tạo Môi trường & GPU

In [ ]:
# @title ⚙️ 1. Cài đặt Môi trường
import os
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

!apt-get install -y -qq aria2 ffmpeg
!pip install -q toml pyyaml python-dotenv bitsandbytes optimum-quanto google-genai openai accelerate safetensors huggingface_hub tqdm pillow av opencv-python-headless diffusers open_clip_torch timm

if not os.path.exists('/content/TranningLoras'):
    !git clone https://github.com/nguyenducvuongg/TranningLoras.git /content/TranningLoras
else:
    !git -C /content/TranningLoras pull

%cd /content/TranningLoras
!pip install -q -e .

from lora_trainer.core.hardware import detect_hardware_environment, setup_cuda_environment
from lora_trainer.storage.drive_manager import setup_storage_structure

setup_storage_structure()
setup_cuda_environment()
hw = detect_hardware_environment()
print(f"🚀 GPU: {hw['gpu_name']} | VRAM: {hw['vram_gb']} GB")

### 🎬 Bước 2: Cấu hình Video Dataset & Huấn luyện

In [ ]:
# @title 🚀 2. Thiết lập Tham số Video & Bắt đầu Huấn luyện
Video_Folder = "/content/drive/MyDrive/TranningLorasData/datasets/train_data/my_video_dataset" # @param {type:'string'}
Model_Type = "Wan2.1-T2V-14B" # @param ["Wan2.1-T2V-14B", "Wan2.1-I2V-14B-720P", "Wan2.1-I2V-14B-480P", "Wan2.1-T2V-1.3B", "Wan2.2-T2V-14B", "Wan2.2-I2V-14B"]

Output_Directory = "/content/drive/MyDrive/TranningLorasData/outputs" # @param {type:'string'}
LoRA_Name = "wan_video_lora" # @param {type:'string'}

Resolution = "720,1280" # @param {type:'string'}
Target_Frames = 25 # @param {type:'integer'}
Frame_Stride = 1 # @param {type:'integer'}
Learning_Rate = 1e-4 # @param {type:'number'}
Network_Dim = 32 # @param {type:'integer'}
Network_Alpha = 16 # @param {type:'integer'}
Max_Train_Epochs = 15 # @param {type:'integer'}
Save_Every_N_Epochs = 1 # @param {type:'integer'}
Sample_Every_N_Steps = 200 # @param {type:'integer'}
Sample_Prompt = "" # @param {type:'string'}
Auto_Disconnect = False # @param {type:'boolean'}

from lora_trainer.engines.unified_trainer import run_unified_training
from lora_trainer.utils.colab_env import auto_disconnect

success = run_unified_training(
    model_name=Model_Type,
    train_folders=Video_Folder,
    output_dir=Output_Directory,
    output_name=LoRA_Name,
    resolution=Resolution,
    learning_rate=Learning_Rate,
    network_dim=Network_Dim,
    network_alpha=Network_Alpha,
    max_train_epochs=Max_Train_Epochs,
    save_every_n_epochs=Save_Every_N_Epochs,
    sample_every_n_steps=Sample_Every_N_Steps,
    sample_prompt=Sample_Prompt,
)

if success and Auto_Disconnect:
    auto_disconnect(force=True)